In [151]:
import torch
import torch.nn as nn

In [152]:
class SelfAttention(nn.Module):

    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()

        self.embed_size = embed_size
        self.heads = heads

        # Dimension handled by each head
        self.head_dim = embed_size // heads

        # Ensure embedding size divisible by heads
        assert (
            self.head_dim * heads == embed_size
        ), "Embed size needs to be divisible by heads"

        # Linear layers for:
        # Values
        # Keys
        # Queries
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)

        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)

        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)

        # Final fully connected layer
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):

        # Batch size
        N = query.shape[0]

        # Lengths
        value_len = values.shape[1]
        key_len = keys.shape[1]
        query_len = query.shape[1]

        # Split embedding into self.heads pieces
        values = values.reshape(
            N,
            value_len,
            self.heads,
            self.head_dim
        )

        keys = keys.reshape(
            N,
            key_len,
            self.heads,
            self.head_dim
        )

        queries = query.reshape(
            N,
            query_len,
            self.heads,
            self.head_dim
        )

        # Generate V, K, Q
        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        # Attention score
        # QK^T

        energy = torch.einsum(
            "nqhd,nkhd->nhqk",
            [queries, keys]
        )

        # nqhd:
        # n = batch size
        # q = query length
        # h = heads
        # d = head dimension

        # nkhd:
        # n = batch size
        # k = key length
        # h = heads
        # d = head dimension

        # Apply mask if available
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        # Softmax
        attention = torch.softmax(
            energy / (self.embed_size ** (1 / 2)),
            dim=3
        )

        # attention shape:
        # (N, heads, query_len, key_len)

        # Multiply attention with values
        out = torch.einsum(
            "nhql,nlhd->nqhd",
            [attention, values]
        )

        # Reshape back
        out = out.reshape(
            N,
            query_len,
            self.heads * self.head_dim
        )

        # Final linear layer
        out = self.fc_out(out)

        return out

In [153]:
class TransformerBlock(nn.Module):

    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()

        self.attention = SelfAttention(embed_size, heads)

        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(

            nn.Linear(embed_size, forward_expansion * embed_size),

            nn.ReLU(),

            nn.Linear(forward_expansion * embed_size, embed_size)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):

        attention = self.attention(value, key, query, mask)

        # Skip connection + normalization
        x = self.dropout(
            self.norm1(attention + query)
        )

        # Feed forward network
        forward = self.feed_forward(x)

        # Another skip connection
        out = self.dropout(
            self.norm2(forward + x)
        )

        return out


In [154]:
class Encoder(nn.Module):

    def __init__(
        self,
        src_vocab_size,
        embed_size,
        num_layers,
        heads,
        device,
        forward_expansion,
        dropout,
        max_length
    ):

        super(Encoder, self).__init__()

        self.embed_size = embed_size
        self.device = device

        # Word embedding
        self.word_embedding = nn.Embedding(
            src_vocab_size,
            embed_size
        )

        # Position embedding
        self.position_embedding = nn.Embedding(
            max_length,
            embed_size
        )

        # Stack encoder layers
        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    embed_size,
                    heads,
                    dropout,
                    forward_expansion
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):

        N, seq_length = x.shape

        positions = torch.arange(
            0,
            seq_length
        ).expand(N, seq_length).to(self.device)

        out = self.dropout(
            self.word_embedding(x)
            + self.position_embedding(positions)
        )

        for layer in self.layers:
            out = layer(out, out, out, mask)

        return out

In [155]:
class DecoderBlock(nn.Module):

    def __init__(self, embed_size, heads, forward_expansion, dropout, device):
        super(DecoderBlock, self).__init__()

        self.norm = nn.LayerNorm(embed_size)

        self.attention = SelfAttention(embed_size, heads)

        self.transformer_block = TransformerBlock(
            embed_size,
            heads,
            dropout,
            forward_expansion
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):

        attention = self.attention(x, x, x, trg_mask)

        query = self.dropout(
            self.norm(attention + x)
        )

        out = self.transformer_block(
            value,
            key,
            query,
            src_mask
        )

        return out

In [156]:
class Decoder(nn.Module):

    def __init__(
        self,
        trg_vocab_size,
        embed_size,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        device,
        max_length
    ):

        super(Decoder, self).__init__()

        self.device = device

        self.word_embedding = nn.Embedding(
            trg_vocab_size,
            embed_size
        )

        self.position_embedding = nn.Embedding(
            max_length,
            embed_size
        )

        self.layers = nn.ModuleList(
            [
                DecoderBlock(
                    embed_size,
                    heads,
                    forward_expansion,
                    dropout,
                    device
                )
                for _ in range(num_layers)
            ]
        )

        self.fc_out = nn.Linear(embed_size, trg_vocab_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):

        N, seq_length = x.shape

        positions = torch.arange(
            0,
            seq_length
        ).expand(N, seq_length).to(self.device)

        x = self.dropout(
            self.word_embedding(x)
            + self.position_embedding(positions)
        )

        for layer in self.layers:

            x = layer(
                x,
                enc_out,
                enc_out,
                src_mask,
                trg_mask
            )

        out = self.fc_out(x)

        return out

In [157]:
class Transformer(nn.Module):

    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        embed_size=512,
        num_layers=6,
        forward_expansion=4,
        heads=8,
        dropout=0,
        device="cpu",
        max_length=100
    ):

        super(Transformer, self).__init__()

        self.encoder = Encoder(
            src_vocab_size,
            embed_size,
            num_layers,
            heads,
            device,
            forward_expansion,
            dropout,
            max_length
        )

        self.decoder = Decoder(
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length
        )

        self.src_pad_idx = src_pad_idx
        self.device = device

    def make_src_mask(self, src):

        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

        return src_mask.to(self.device)

    def make_trg_mask(self, trg):

        N, trg_len = trg.shape

        trg_mask = torch.tril(
            torch.ones((trg_len, trg_len))
        ).expand(N, 1, trg_len, trg_len)

        return trg_mask.to(self.device)

    def forward(self, src, trg):

        src_mask = self.make_src_mask(src)

        trg_mask = self.make_trg_mask(trg)

        enc_src = self.encoder(src, src_mask)

        out = self.decoder(
            trg,
            enc_src,
            src_mask,
            trg_mask
        )

        return out

In [158]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sample parameters
src_vocab_size = 10000
trg_vocab_size = 10000

src_pad_idx = 0

model = Transformer(
    src_vocab_size,
    trg_vocab_size,
    src_pad_idx,
    embed_size=512,
    num_layers=6,
    forward_expansion=4,
    heads=8,
    dropout=0,
    device=device,
    max_length=100
).to(device)

In [159]:
x = torch.tensor([
    [1, 5, 6, 4, 3, 9, 5, 2, 0],
    [1, 8, 7, 3, 4, 5, 6, 7, 2]
]).to(device)

trg = torch.tensor([
    [1, 7, 4, 3, 5, 9, 2],
    [1, 5, 6, 2, 4, 7, 6]
]).to(device)

In [160]:
out = model(x, trg[:, :-1])

print(out.shape)

torch.Size([2, 6, 10000])


In [161]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [162]:
vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "i": 3,
    "love": 4,
    "deep": 5,
    "learning": 6,
    "transformers": 7,
    "are": 8,
    "awesome": 9
}


In [163]:
idx_to_word = {v: k for k, v in vocab.items()}

In [164]:
src = torch.tensor([
    [1, 3, 4, 7, 2, 0, 0]
]).to(device)

In [165]:
trg = torch.tensor([
    [1, 7, 8, 9, 2]
]).to(device)

In [166]:
model = Transformer(
    src_vocab_size=len(vocab),
    trg_vocab_size=len(vocab),
    src_pad_idx=vocab["<PAD>"],
    embed_size=64,
    num_layers=2,
    forward_expansion=4,
    heads=8,
    dropout=0.1,
    device=device,
    max_length=100
).to(device)


In [167]:
output = model(src, trg[:, :-1])

print("Output shape:")
print(output.shape)

# Shape:
# (batch_size, target_length, vocab_size)

Output shape:
torch.Size([1, 4, 10])


In [168]:
predictions = torch.argmax(output, dim=2)

print("\nPredicted token IDs:")
print(predictions)


Predicted token IDs:
tensor([[6, 8, 7, 9]])


In [169]:
print("\nPredicted words:")

for sentence in predictions:

    words = []

    for token in sentence:
        token_id = token.item()

        word = idx_to_word.get(token_id, "<UNK>")

        words.append(word)

    print(" ".join(words))


Predicted words:
learning are transformers awesome
